In [2]:
import pandas as pd
import ast
import re
import numpy as np

# Load the original dataset
df = pd.read_csv('recipes_final.csv', low_memory=False)

# Create a new dataframe with 'Name' column
result_df = pd.DataFrame()
result_df['Name'] = df['Name']

# Function to clean and extract steps from the RecipeInstructions column
def extract_steps(instructions_str):
    # Check if the value is NaN
    if pd.isna(instructions_str):
        return []
    
    try:
        # Convert string representation of list to actual list
        instructions_list = ast.literal_eval(instructions_str)
        
        # Return the list of steps
        return instructions_list
    except:
        # If there's an error in parsing, try a different approach
        try:
            # This handles cases where the format might be inconsistent
            steps = re.findall(r"'([^']*)'", str(instructions_str))
            return steps if steps else []
        except:
            # If all else fails, return empty list
            return []

# Maximum number of steps (as specified)
max_steps = 25

# Process each row to extract steps
for i, row in df.iterrows():
    steps = extract_steps(row['RecipeInstructions'])
    
    # Add each step to the result dataframe
    for step_num, step in enumerate(steps, 1):
        if step_num <= max_steps:  # Only process up to max_steps
            col_name = f'step-{step_num}'
            if col_name not in result_df.columns:
                result_df[col_name] = None  # Create column if it doesn't exist
            result_df.at[i, col_name] = step

# Fill NaN values with empty strings
result_df = result_df.fillna('')

# Save the result to a new CSV file
result_df.to_csv('recipes_steps_separated.csv', index=False)

print(f"Processed {len(df)} recipes with up to {max_steps} steps per recipe.")
print("Output saved to 'recipes_steps_separated.csv'")


Processed 9996 recipes with up to 25 steps per recipe.
Output saved to 'recipes_steps_separated.csv'


In [5]:
import pandas as pd
import re

# Load the CSV file
df = pd.read_csv('recipes_steps_separated.csv')

# Create a new dataframe for the results
result_df = pd.DataFrame()
result_df['Name'] = df['Name']

# Maximum number of steps to create
max_steps = 50  # Adjust if needed

# Function to split text by periods and semicolons
def split_text(text):
    if pd.isna(text) or text == '':
        return []
    
    # Convert to string if not already
    text = str(text)
    
    # Split by both period and semicolon
    # Using regex to avoid splitting decimal numbers
    splits = re.split(r'(?<!\d)\.(?!\d)|;', text)
    
    # Clean up each split
    return [split.strip() for split in splits if split.strip()]

# Process each row and column
step_count = 0
for i, row in df.iterrows():
    step_count = 0  # Reset step count for each recipe
    
    # Process each step column
    for col in df.columns:
        if col == 'Name':
            continue
        
        cell_content = row[col]
        if pd.notna(cell_content) and cell_content != '':
            # Split the cell content
            sub_steps = split_text(cell_content)
            
            # Add each sub-step as a new column
            for sub_step in sub_steps:
                if sub_step:  # Only add non-empty sub-steps
                    step_count += 1
                    step_col = f'step-{step_count}'
                    
                    # Create the column if it doesn't exist
                    if step_col not in result_df.columns:
                        result_df[step_col] = ""
                    
                    # Add the sub-step to the appropriate row
                    result_df.at[i, step_col] = sub_step

# Fill NaN values with empty strings
result_df = result_df.fillna('')

# Save the result to a new CSV file
result_df.to_csv('recipes_split_by_periods_semicolons.csv', index=False)

print("Processing complete. Output saved to 'recipes_split_by_periods_semicolons.csv'")


Processing complete. Output saved to 'recipes_split_by_periods_semicolons.csv'


In [6]:
import pandas as pd
import random

# Load the CSV file
df = pd.read_csv('recipes_split_by_periods_semicolons.csv')

# Create a new dataframe with the same column names
result_df = pd.DataFrame(columns=df.columns)

# Keep the 'Name' column the same
result_df['Name'] = df['Name']

# For each row, shuffle the step columns
for i, row in df.iterrows():
    # Get all columns except 'Name'
    step_columns = [col for col in df.columns if col != 'Name']
    
    # Get the values for these columns
    step_values = [row[col] for col in step_columns]
    
    # Shuffle the values
    random.shuffle(step_values)
    
    # Assign the shuffled values back to the result dataframe
    for j, col in enumerate(step_columns):
        result_df.at[i, col] = step_values[j]

# Fill NaN values with empty strings
result_df = result_df.fillna('')

# Save the result to a new CSV file
result_df.to_csv('recipes_jumbled_steps.csv', index=False)

print("Processing complete. Output saved to 'recipes_jumbled_steps.csv'")


Processing complete. Output saved to 'recipes_jumbled_steps.csv'


In [7]:
import pandas as pd

# Load the CSV file with jumbled steps
df = pd.read_csv('recipes_jumbled_steps.csv')

# Create a new dataframe for the cleaned data
result_df = pd.DataFrame()
result_df['Name'] = df['Name']

# For each row, collect all non-empty cells and reindex them
for i, row in df.iterrows():
    # Get all non-empty values except the 'Name' column
    non_empty_values = [val for col, val in row.items() 
                       if col != 'Name' and pd.notna(val) and val != '']
    
    # Add these values to the result dataframe with sequential column names
    for j, value in enumerate(non_empty_values):
        col_name = f'step-{j+1}'
        
        # Create the column if it doesn't exist
        if col_name not in result_df.columns:
            result_df[col_name] = None
        
        # Add the value to the appropriate cell
        result_df.at[i, col_name] = value

# Fill any remaining NaN values with empty strings
result_df = result_df.fillna('')

# Save the result to a new CSV file
result_df.to_csv('recipes_cleaned_no_gaps.csv', index=False)

print("Empty cells removed. Output saved to 'controlflow_recipies.csv'")


Empty cells removed. Output saved to 'recipes_cleaned_no_gaps.csv'


In [11]:
import pandas as pd
import random

# Load the dataset
df = pd.read_csv('controlflow_recipies.csv')

# Create a dictionary to categorize recipes
recipe_categories = {
    'Dessert': ['Scones', 'Biscuits', 'Cake', 'Cookie', 'Bread', 'Pie', 'Crepes'],
    'Meat': ['Beef', 'Chicken', 'Pork', 'Lamb', 'Steak', 'Meatball'],
    'Seafood': ['Fish', 'Salmon', 'Crab', 'Shrimp', 'Lobster'],
    'Vegetable': ['Spinach', 'Broccoli', 'Potato', 'Tomato', 'Zucchini', 'Cauliflower'],
    'Soup': ['Soup', 'Chowder', 'Stew'],
    'Sauce': ['Sauce', 'Dip', 'Gravy']
}

# Categorize each recipe
def categorize_recipe(recipe_name):
    for category, keywords in recipe_categories.items():
        for keyword in keywords:
            if keyword.lower() in recipe_name.lower():
                return category
    return 'Other'

# Add category to each recipe
df['Category'] = df['Name'].apply(categorize_recipe)

# Create a copy for the modified dataset (after adding the Category column)
result_df = df.copy()

# Get all step columns (excluding the 'Name' and 'Category' columns)
step_columns = [col for col in df.columns if col != 'Name' and col != 'Category']

# For each recipe, replace 50% of its steps with steps from recipes in different categories
for i, row in df.iterrows():
    # Get the current recipe's category
    current_category = row['Category']
    
    # Find recipes from different categories
    different_category_recipes = df[df['Category'] != current_category]
    
    if len(different_category_recipes) == 0:
        continue  # Skip if no different categories found
    
    # Get non-empty steps from the current recipe
    valid_steps = [col for col in step_columns if pd.notna(row[col]) and row[col] != '']
    
    if not valid_steps:
        continue  # Skip if no valid steps
    
    # Calculate how many steps to replace (50%)
    num_steps_to_replace = max(1, len(valid_steps) // 2)
    
    # Randomly select steps to replace
    steps_to_replace = random.sample(valid_steps, num_steps_to_replace)
    
    # For each step to replace, find a random step from a different category
    for step_col in steps_to_replace:
        # Select a random recipe from a different category
        random_recipe = different_category_recipes.sample(1).iloc[0]
        
        # Find non-empty steps in the random recipe
        random_recipe_steps = [col for col in step_columns 
                              if pd.notna(random_recipe[col]) and random_recipe[col] != '']
        
        if random_recipe_steps:
            # Select a random step
            random_step_col = random.choice(random_recipe_steps)
            
            # Replace the step in the result dataframe
            result_df.at[i, step_col] = random_recipe[random_step_col]

# Remove the temporary category column
result_df.drop('Category', axis=1, inplace=True)

# Save the contaminated dataset
result_df.to_csv('recipes_with_50percent_replaced_steps.csv', index=False)

print("Controlled contamination complete. 50% of steps replaced with steps from different recipe categories.")
print("Output saved to 'dataflow_recipies.csv'") 


Controlled contamination complete. 50% of steps replaced with steps from different recipe categories.
Output saved to 'recipes_with_50percent_replaced_steps.csv'
